# EN3150 Assignment 03 — Shanuka's Notebook
## SOTA Transfer Learning: MobileNetV2 & ShuffleNetV2 on RealWaste 64×64

**Team:** Outliers | **Author:** Shanuka (`@ashanuk`)

---

### 🔄 Workflow: How this notebook connects to the IDE

```
Antigravity IDE  ──git push──▶  GitHub  ──Cell 1 (Sync)──▶  Colab GPU
```

- **Code changes** happen in the IDE → pushed to GitHub
- **Re-run Cell 1** anytime to pull the latest code into this Colab session
- **Training runs** happen here on Colab's GPU
- **Outputs** (figures, weights) are downloaded back and committed from the IDE

> ⚠️ **Before starting:** `Runtime → Change runtime type → T4 GPU → Save`

---
## 🔄 SYNC CELL — Pull Latest Code from GitHub
> **Run this cell any time** your teammate edits code in the IDE and pushes to GitHub.
> It safely pulls the latest `feature/sota-transfer` branch without losing any training outputs.

In [ ]:
import os

REPO_URL = 'https://github.com/yumyum-web/pattern-ass-03.git'
BRANCH   = 'feature/sota-transfer'
REPO_DIR = '/content/pattern-ass-03'

if not os.path.exists(REPO_DIR):
    print('📥 First run — cloning repo...')
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print('🔄 Repo already cloned — pulling latest changes from GitHub...')
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} reset --hard origin/{BRANCH}

os.chdir(REPO_DIR)

# Add repo to Python path so all src/ imports work
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print()
!git log --oneline -5
print(f'\n✅ Synced! Working directory: {os.getcwd()}')

---
## 1. Install Dependencies

In [ ]:
# torch/torchvision are pre-installed on Colab — just install the extras
!pip install -q scikit-learn tqdm ucimlrepo torchinfo tabulate

import torch
import torchvision
print(f'PyTorch     : {torch.__version__}')
print(f'Torchvision : {torchvision.__version__}')
gpu = torch.cuda.is_available()
print(f'GPU         : {torch.cuda.get_device_name(0) if gpu else "❌ NOT AVAILABLE — change runtime type!"}')
if not gpu:
    print('\n⚠️  Go to Runtime → Change runtime type → T4 GPU → Save, then reconnect.')

---
## 2. Dataset — RealWaste 64×64 (UCI ID: 908)

In [ ]:
from src.data import get_realwaste_dataloaders, CLASSES

train_loader, val_loader, test_loader, class_to_idx = get_realwaste_dataloaders(
    data_dir='data', batch_size=32, image_size=64, seed=42, num_workers=2, download=True,
)

total = len(train_loader.dataset) + len(val_loader.dataset) + len(test_loader.dataset)
print(f'Classes : {CLASSES}')
print(f'Train   : {len(train_loader.dataset):,} ({len(train_loader.dataset)/total*100:.1f}%)')
print(f'Val     : {len(val_loader.dataset):,} ({len(val_loader.dataset)/total*100:.1f}%)')
print(f'Test    : {len(test_loader.dataset):,} ({len(test_loader.dataset)/total*100:.1f}%)')

imgs, labels = next(iter(train_loader))
assert imgs.shape[1:] == (3, 64, 64), f'Bad shape: {imgs.shape}'
print(f'Batch   : {tuple(imgs.shape)}  ✅')

---
## 3. SOTA Model Wrappers — C05
> `feat(sota): create MobileNetV2 and ShuffleNetV2 transfer learning wrappers`

In [ ]:
from src.models.sota_models import (
    get_mobilenet_v2, get_shufflenet_v2,
    count_parameters, get_model_size_mb, print_model_summary,
)

mobilenet  = get_mobilenet_v2(num_classes=9, pretrained=True)
shufflenet = get_shufflenet_v2(num_classes=9, pretrained=True, width_mult='1_0')

print_model_summary(mobilenet,  'MobileNetV2')
print_model_summary(shufflenet, 'ShuffleNetV2 ×1.0')

# Sanity check: forward pass with 64x64 input
dummy = torch.zeros(2, 3, 64, 64)
with torch.no_grad():
    assert mobilenet(dummy).shape  == (2, 9)
    assert shufflenet(dummy).shape == (2, 9)
print('\n✅ C05 verified — both wrappers produce (2, 9) output')

---
## 4. Fine-Tuning — C12
> `test(sota): fine-tune MobileNetV2 and ShuffleNetV2 on RealWaste 64x64`

| Setting | Value |
|---------|-------|
| Optimiser | Adam lr=0.001, wd=1e-4 |
| Scheduler | CosineAnnealingLR T_max=20 |
| Loss | CrossEntropyLoss |
| Epochs | 20 per model |
| Batch | 32 |

In [ ]:
import time
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS      = 20
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
WEIGHTS_DIR = 'weights'
FIGURES_DIR = 'figures'
os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
print(f'Training on: {DEVICE}')

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    loss_sum, correct, n = 0.0, 0, 0
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, lbls)
        loss.backward(); optimizer.step()
        loss_sum += loss.item() * imgs.size(0)
        correct  += (out.argmax(1) == lbls).sum().item()
        n        += imgs.size(0)
    return loss_sum / n, correct / n

def eval_epoch(model, loader, criterion):
    model.eval()
    loss_sum, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            out  = model(imgs)
            loss = criterion(out, lbls)
            loss_sum += loss.item() * imgs.size(0)
            correct  += (out.argmax(1) == lbls).sum().item()
            n        += imgs.size(0)
    return loss_sum / n, correct / n

def train_model(model, name, save_path):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
    history   = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_acc  = 0.0

    print(f'\n{"─"*60}')
    print(f'  {name}')
    print(f'{"─"*60}')
    print(f'  {"Ep":>3}  {"TrainLoss":>10}  {"TrainAcc":>9}  {"ValLoss":>9}  {"ValAcc":>8}  {"Sec":>5}')

    for ep in range(1, EPOCHS + 1):
        t0 = time.time()
        tl, ta = train_epoch(model, train_loader, criterion, optimizer)
        vl, va = eval_epoch(model, val_loader, criterion)
        scheduler.step()
        history['train_loss'].append(tl); history['val_loss'].append(vl)
        history['train_acc'].append(ta);  history['val_acc'].append(va)
        print(f'  {ep:>3}  {tl:>10.4f}  {ta*100:>8.2f}%  {vl:>9.4f}  {va*100:>7.2f}%  {time.time()-t0:>4.1f}s')
        if va > best_acc:
            best_acc = va
            torch.save(model.state_dict(), save_path)

    print(f'\n  ✅ Best val acc: {best_acc*100:.2f}% → {save_path}')
    return model, history

print('Helper functions ready ✅')

In [ ]:
# ── Train MobileNetV2  (~10 min on T4) ────────────────────────────────────
mobilenet, hist_mn = train_model(
    get_mobilenet_v2(num_classes=9, pretrained=True),
    name='MobileNetV2',
    save_path=f'{WEIGHTS_DIR}/mobilenet_v2.pth',
)

In [ ]:
# ── Train ShuffleNetV2 (~10 min on T4) ────────────────────────────────────
shufflenet, hist_sn = train_model(
    get_shufflenet_v2(num_classes=9, pretrained=True, width_mult='1_0'),
    name='ShuffleNetV2-1.0×',
    save_path=f'{WEIGHTS_DIR}/shufflenet_v2.pth',
)

In [ ]:
# ── Plot & save training curves ───────────────────────────────────────────
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

histories = {'MobileNetV2': hist_mn, 'ShuffleNetV2-1.0×': hist_sn}
eps       = range(1, EPOCHS + 1)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for row, (name, h) in enumerate(histories.items()):
    al, aa = axes[row]
    al.plot(eps, h['train_loss'], label='Train', color='#2196F3', lw=1.8)
    al.plot(eps, h['val_loss'],   label='Val',   color='#F44336', lw=1.8, ls='--')
    al.set_title(f'{name} — Loss', fontweight='bold'); al.legend(); al.grid(alpha=0.3)
    al.set_xlabel('Epoch'); al.set_ylabel('Loss')

    aa.plot(eps, [a*100 for a in h['train_acc']], label='Train', color='#4CAF50', lw=1.8)
    aa.plot(eps, [a*100 for a in h['val_acc']],   label='Val',   color='#FF9800', lw=1.8, ls='--')
    aa.set_title(f'{name} — Accuracy', fontweight='bold'); aa.legend(); aa.grid(alpha=0.3)
    aa.set_xlabel('Epoch'); aa.set_ylabel('Accuracy (%)'); aa.set_ylim(0, 100)

fig.suptitle('SOTA Transfer Learning — Training Curves\nRealWaste 64×64', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/sota_loss_curves.pdf', bbox_inches='tight', dpi=150)
plt.show()
print('✅ C12 complete — sota_loss_curves.pdf saved')

---
## 5. Test Set Evaluation — C14
> `feat(eval): evaluate SOTA models on test set with confusion matrices and precision-recall`

In [ ]:
from src.utils.metrics import (
    collect_predictions, compute_metrics, print_metrics,
    save_confusion_matrices, plot_confusion_matrix, CLASSES as M_CLASSES,
)

configs = [
    {'name': 'MobileNetV2',       'model': mobilenet,  'path': f'{WEIGHTS_DIR}/mobilenet_v2.pth'},
    {'name': 'ShuffleNetV2-1.0×', 'model': shufflenet, 'path': f'{WEIGHTS_DIR}/shufflenet_v2.pth'},
]

confusion_data = []
rows           = []

for cfg in configs:
    cfg['model'].load_state_dict(torch.load(cfg['path'], map_location=DEVICE))
    cfg['model'] = cfg['model'].to(DEVICE)
    preds, targets = collect_predictions(cfg['model'], test_loader, DEVICE)
    print_metrics(preds, targets, model_name=cfg['name'])
    acc, prec, rec = compute_metrics(preds, targets)
    trainable, _   = count_parameters(cfg['model'])
    size_mb        = get_model_size_mb(cfg['model'])
    confusion_data.append((cfg['name'], preds, targets))
    rows.append({'Model': cfg['name'], 'Params': f'{trainable:,}', 'Size(MB)': f'{size_mb:.2f}',
                 'TestAcc': f'{acc*100:.2f}%', 'MacroPrec': f'{prec*100:.2f}%', 'MacroRec': f'{rec*100:.2f}%'})

In [ ]:
# ── Confusion matrices (saved + shown inline) ─────────────────────────────
save_confusion_matrices(confusion_data, save_path=f'{FIGURES_DIR}/confusion_matrices.pdf')

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for i, (name, preds, targets) in enumerate(confusion_data):
    plot_confusion_matrix(preds, targets, model_name=name, ax=axes[i])
plt.suptitle('Normalised Confusion Matrices — RealWaste 64×64 Test Set',
             fontweight='bold', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()
print('✅ C14 complete — confusion_matrices.pdf saved')

In [ ]:
# ── Final comparison table ────────────────────────────────────────────────
import pandas as pd
df = pd.DataFrame(rows)
print('\n=== SOTA COMPARISON TABLE ===')
print(df.to_markdown(index=False))
display(df)

---
## 6. 💾 Download Outputs & Commit

Run the cell below to download all outputs. Then in the IDE terminal:

```bash
# Place downloaded files in the repo, then:
git add figures/sota_loss_curves.pdf figures/confusion_matrices.pdf weights/
git commit -m "test(sota): fine-tune MobileNetV2 and ShuffleNetV2 on RealWaste 64x64"

git add figures/confusion_matrices.pdf
git commit -m "feat(eval): evaluate SOTA models on test set with confusion matrices and precision-recall"

git push origin feature/sota-transfer
```

In [ ]:
# ── Download everything back to your computer ─────────────────────────────
from google.colab import files
import glob

outputs = glob.glob('figures/*.pdf') + glob.glob('weights/*.pth')
print(f'Downloading {len(outputs)} files:')
for f in outputs:
    print(f'  📄 {f}')
    files.download(f)
print('\n✅ All outputs downloaded — now commit them from the IDE!')